# 09 — Capstone: Stock Forecasting with RevIN

## Project overview

This notebook ties everything together into a complete forecasting project:

1. Load stock data
2. Train a **baseline model** (no RevIN)
3. Train a **RevIN model** using per-instance normalization
4. Evaluate both on the **same-distribution** test set
5. Evaluate both on the **shifted-distribution** dataset
6. Visualise and compare predictions

## Hypothesis

The baseline model is trained on globally-normalised data and will struggle
when tested on the shifted dataset (different price regime).
The RevIN model normalizes each window independently, so it should be
**more robust** to the distribution shift.

---

In [ ]:
import sys, pathlib, pickle
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

from utils.ml_utils import (
    RevIN, z_score_normalize, denormalize_data,
    create_sequences, train_test_split_time_series,
    mse, mae, rmse
)
from utils.plotting import plot_line, plot_before_after_distribution

## 1 — Load data

In [ ]:
df_orig    = pd.read_csv('../data/stock_sample.csv')
df_shifted = pd.read_csv('../data/shifted_stock_sample.csv')

close_orig    = df_orig['close'].values.astype(float)
close_shifted = df_shifted['close'].values.astype(float)

print(f'Original  — {len(close_orig)} rows, mean={close_orig.mean():.2f}, std={close_orig.std():.2f}')
print(f'Shifted   — {len(close_shifted)} rows, mean={close_shifted.mean():.2f}, std={close_shifted.std():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(close_orig,    color='#4C72B0'); axes[0].set_title('Original Stock Price'); axes[0].set_ylabel('Price ($)')
axes[1].plot(close_shifted, color='#DD8452'); axes[1].set_title('Shifted Stock Price (different regime)')
for ax in axes:
    ax.set_xlabel('Day'); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 2 — Baseline model (global normalization, no RevIN)

In [ ]:
LOOKBACK = 20
HORIZON  = 1

# Global normalization using training data statistics
close_norm, norm_params = z_score_normalize(close_orig)

X, y = create_sequences(close_norm, lookback=LOOKBACK, horizon=HORIZON)
y    = y.ravel()
X_tr, X_te, y_tr, y_te = train_test_split_time_series(X, y, test_ratio=0.2)

baseline = RandomForestRegressor(n_estimators=100, random_state=42)
baseline.fit(X_tr, y_tr)
print('Baseline model trained.')

In [ ]:
# Evaluate on same-distribution test set
y_pred_norm = baseline.predict(X_te)
y_te_orig   = denormalize_data(y_te,        norm_params)
y_pred_orig = denormalize_data(y_pred_norm, norm_params)

print('=== Baseline — Same Distribution ===')
print(f'  RMSE : {rmse(y_te_orig, y_pred_orig):.4f}')
print(f'  MAE  : {mae(y_te_orig,  y_pred_orig):.4f}')

In [ ]:
# Evaluate baseline on SHIFTED distribution (bad — using wrong global stats)
close_shifted_norm = (close_shifted - norm_params['mean']) / norm_params['std']
X_s, y_s = create_sequences(close_shifted_norm, lookback=LOOKBACK, horizon=HORIZON)
y_s = y_s.ravel()

y_s_pred_norm = baseline.predict(X_s)
y_s_orig      = denormalize_data(y_s,          norm_params)
y_s_pred_orig = denormalize_data(y_s_pred_norm, norm_params)

baseline_rmse_shift = rmse(y_s_orig, y_s_pred_orig)
baseline_mae_shift  = mae(y_s_orig,  y_s_pred_orig)
print('=== Baseline — Shifted Distribution ===')
print(f'  RMSE : {baseline_rmse_shift:.4f}')
print(f'  MAE  : {baseline_mae_shift:.4f}')

## 3 — RevIN model

In [ ]:
def build_revin_sequences(series, lookback, horizon):
    """Create sequences and apply per-instance RevIN normalization."""
    X_raw, y_raw = create_sequences(series, lookback, horizon)
    y_raw = y_raw.ravel()
    X_norm, y_norm = np.zeros_like(X_raw), np.zeros_like(y_raw)
    revins = []
    for i in range(len(X_raw)):
        r = RevIN()
        X_norm[i] = r.normalize(X_raw[i])
        y_norm[i] = (y_raw[i] - r.stats['mean']) / r.stats['std']
        revins.append(r)
    return X_norm, y_norm, X_raw, y_raw, revins


X_rn, y_rn, X_raw, y_raw, revins_tr = build_revin_sequences(close_orig, LOOKBACK, HORIZON)
Xr_tr, Xr_te, yr_tr, yr_te, _, _, r_tr, r_te = (
    *train_test_split_time_series(X_rn, y_rn, 0.2),
    *train_test_split_time_series(X_raw, y_raw, 0.2)
)

revin_model = RandomForestRegressor(n_estimators=100, random_state=42)
revin_model.fit(Xr_tr, yr_tr)
print('RevIN model trained.')

In [ ]:
def predict_with_revin(model, X_raw_windows, y_true):
    """Predict each window: normalize in, denormalize out."""
    preds = []
    for i in range(len(X_raw_windows)):
        r      = RevIN()
        x_norm = r.normalize(X_raw_windows[i])
        y_norm = model.predict(x_norm.reshape(1, -1))[0]
        y_hat  = r.denormalize(np.array([y_norm]))[0]
        preds.append(y_hat)
    return np.array(preds)


X_raw_all, y_raw_all = create_sequences(close_orig, LOOKBACK, HORIZON)
y_raw_all = y_raw_all.ravel()
_, X_te_raw, _, y_te_raw = train_test_split_time_series(X_raw_all, y_raw_all, 0.2)

y_rev_pred = predict_with_revin(revin_model, X_te_raw, y_te_raw)

print('=== RevIN — Same Distribution ===')
print(f'  RMSE : {rmse(y_te_raw, y_rev_pred):.4f}')
print(f'  MAE  : {mae(y_te_raw,  y_rev_pred):.4f}')

In [ ]:
# RevIN on shifted distribution
X_s_raw, y_s_raw = create_sequences(close_shifted, LOOKBACK, HORIZON)
y_s_raw = y_s_raw.ravel()

y_rev_shifted = predict_with_revin(revin_model, X_s_raw, y_s_raw)

revin_rmse_shift = rmse(y_s_raw, y_rev_shifted)
revin_mae_shift  = mae(y_s_raw,  y_rev_shifted)
print('=== RevIN — Shifted Distribution ===')
print(f'  RMSE : {revin_rmse_shift:.4f}')
print(f'  MAE  : {revin_mae_shift:.4f}')

## 4 — Results comparison

In [ ]:
results = {
    'Model':    ['Baseline', 'Baseline', 'RevIN', 'RevIN'],
    'Dataset':  ['Same dist', 'Shifted', 'Same dist', 'Shifted'],
    'RMSE':     [
        rmse(y_te_orig, y_pred_orig),
        baseline_rmse_shift,
        rmse(y_te_raw,  y_rev_pred),
        revin_rmse_shift
    ],
    'MAE': [
        mae(y_te_orig, y_pred_orig),
        baseline_mae_shift,
        mae(y_te_raw,  y_rev_pred),
        revin_mae_shift
    ]
}

results_df = pd.DataFrame(results)
results_df['RMSE'] = results_df['RMSE'].round(4)
results_df['MAE']  = results_df['MAE'].round(4)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

groups    = ['Same dist', 'Shifted']
x         = np.arange(len(groups))
width     = 0.35

for ax, metric in zip(axes, ['RMSE', 'MAE']):
    baseline_vals = results_df[results_df['Model'] == 'Baseline'][metric].values
    revin_vals    = results_df[results_df['Model'] == 'RevIN'][metric].values

    bars_b = ax.bar(x - width/2, baseline_vals, width, label='Baseline', color='#4C72B0', alpha=0.85)
    bars_r = ax.bar(x + width/2, revin_vals,    width, label='RevIN',    color='#55A868', alpha=0.85)

    for bar in [*bars_b, *bars_r]:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    ax.set_title(f'{metric} — Baseline vs RevIN', fontsize=13)
    ax.set_ylabel(metric)
    ax.legend()
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('RevIN degrades more gracefully under distribution shift', fontsize=14)
plt.tight_layout()
plt.show()

## 5 — Visualise predictions vs actuals

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
t = np.arange(len(y_te_raw))

# Same distribution
axes[0].plot(t, y_te_raw,     color='black',   linewidth=1.5, label='Actual', alpha=0.7)
axes[0].plot(t, y_pred_orig,  color='#4C72B0', linewidth=1.2, label='Baseline', linestyle='--')
axes[0].plot(t, y_rev_pred,   color='#55A868', linewidth=1.2, label='RevIN',    linestyle='-.')
axes[0].set_title('Predictions on Same Distribution', fontsize=12)
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Shifted distribution
t_s = np.arange(len(y_s_raw))
axes[1].plot(t_s, y_s_raw,       color='black',   linewidth=1.5, label='Actual', alpha=0.7)
axes[1].plot(t_s, y_s_pred_orig, color='#4C72B0', linewidth=1.2, label='Baseline', linestyle='--')
axes[1].plot(t_s, y_rev_shifted, color='#55A868', linewidth=1.2, label='RevIN',    linestyle='-.')
axes[1].set_title('Predictions on Shifted Distribution (new regime)', fontsize=12)
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Price ($)')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Forecast Quality: Baseline vs RevIN', fontsize=14)
plt.tight_layout()
plt.show()

## 6 — Load pre-trained models

In [ ]:
with open('../models/baseline_model.pkl', 'rb') as f:
    bl_pack = pickle.load(f)
    
with open('../models/revin_model.pkl', 'rb') as f:
    rv_pack = pickle.load(f)

print('Baseline model lookback:', bl_pack['lookback'])
print('RevIN    model lookback:', rv_pack['lookback'])
print('Both models loaded successfully.')

## 7 — Key takeaways

| Lesson | Detail |
|--------|--------|
| **Global normalization** | Uses a fixed μ and σ from training data — fails when test distribution shifts |
| **RevIN normalize** | Computes μ and σ freshly for each input window — always zero-mean, unit-std |
| **RevIN denormalize** | Multiplies prediction by σ and adds μ to get back original scale |
| **When it helps most** | Regime changes, non-stationary series, long-horizon forecasting |
| **Trade-off** | Slightly more computation; removes global trend info from each window |

---
## Final exercises

1. Try `LOOKBACK = 5` and `LOOKBACK = 40`. How does window size affect RevIN's advantage?
2. Can you add learnable affine parameters γ and β to RevIN (like Layer Norm)
   so the model can partially restore the original scale distribution?
3. What would happen if you applied RevIN to the `volume` column instead of
   `close`? Would it still help?
4. Read the original RevIN paper (Kim et al., 2022, ICLR). Compare their
   implementation to ours — what extra components do they include?

---
**Congratulations — you've completed the full RevIN learning path!**

```
mean → variance → std → normalization → z-score
  → time series → distribution shift → RevIN → forecasting project
```